# Evaluate FLAN-T5-base baseline on MedQA with full fine-tuning
Files needed:
*   MedQA data (train, valid, test)


Make sure to define the path to the MedQA data

In [ ]:
path_to_train = "./medqa_data/US/train.jsonl"
path_to_dev = "./medqa_data/US/dev.jsonl"
path_to_test = "./medqa_data/US/test.jsonl"

In [ ]:
! pip3 install transformers datasets torch accelerate evaluate

## Load base data

One thing to note about the **meta_info** feature :

step 1 ---> basic science

step 2 & 3 ---> clinical knowledge

In [ ]:
import pandas as pd
import os

train_df = pd.read_json(path_to_train, lines=True)
dev_df = pd.read_json(path_to_dev, lines=True)
test_df = pd.read_json(path_to_test, lines=True)

In [ ]:
train_df.info()
dev_df.info()
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10178 entries, 0 to 10177
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    10178 non-null  object
 1   answer      10178 non-null  object
 2   options     10178 non-null  object
 3   meta_info   10178 non-null  object
 4   answer_idx  10178 non-null  object
dtypes: object(5)
memory usage: 397.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1272 entries, 0 to 1271
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    1272 non-null   object
 1   answer      1272 non-null   object
 2   options     1272 non-null   object
 3   meta_info   1272 non-null   object
 4   answer_idx  1272 non-null   object
dtypes: object(5)
memory usage: 49.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1273 entries, 0 to 1272
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtyp

In [ ]:
train_df.head()

,question,answer,options,meta_info,answer_idx
0,A 23-year-old pregnant woman at 22 weeks gesta...,Nitrofurantoin,"{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': '...",step2&3,E
1,A 3-month-old baby died suddenly at night whil...,Placing the infant in a supine position on a f...,{'A': 'Placing the infant in a supine position...,step2&3,A
2,A mother brings her 3-week-old infant to the p...,Abnormal migration of ventral pancreatic bud,{'A': 'Abnormal migration of ventral pancreati...,step1,A
3,A pulmonary autopsy specimen from a 58-year-ol...,Thromboembolism,"{'A': 'Thromboembolism', 'B': 'Pulmonary ische...",step1,A
4,A 20-year-old woman presents with menorrhagia ...,Von Willebrand disease,"{'A': 'Factor V Leiden', 'B': 'Hemophilia A', ...",step1,E


# Now let's get serious

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
def preprocess_medqa_t5(hf_dataset):
    inputs = []

    for example in hf_dataset:
        question = example["question"]
        options = [f"{k}: {v}" for k, v in example["options"].items()]
        #answer = example["options"][example["answer_idx"]]
        answer = example["answer_idx"]

        # FLAN-T5 expects text-to-text input format
        prompt = f"question: {question} options: {', '.join(options)}"

        encoding = tokenizer(prompt, padding="max_length", truncation=True, max_length=512)
        label_encoding = tokenizer(answer, padding="max_length", truncation=True, max_length=2)

        inputs.append({
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "labels": label_encoding["input_ids"]
        })

    return Dataset.from_list(inputs)

In [ ]:
import torch
from datasets import Dataset


# Convert DataFrame to Hugging Face Dataset
hf_train = Dataset.from_pandas(train_df)
# Apply tokenization
train_inputs = preprocess_medqa_t5(hf_train)

hf_valid = Dataset.from_pandas(dev_df)
valid_inputs = preprocess_medqa_t5(hf_valid)

In [ ]:
# Verifying if we are working on the GPU
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Number of GPUs available
print(torch.cuda.get_device_name(0))  # GPU name
model.to("cuda")
print(next(model.parameters()).device)  # Should return: cuda:0

True
1
Tesla T4
cuda:0


In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

def compute_metrics_t5(eval_pred):
    preds, labels = eval_pred

    # Handle tuple structure
    if isinstance(preds, tuple):
        preds = preds[0]  # Use the token IDs only

    # Select the most probable token at each position
    if len(preds.shape) == 3:  # Shape: (batch_size, seq_length, vocab_size)
        preds = np.argmax(preds, axis=-1)  # Shape becomes (batch_size, seq_length)

    # Ensure preds and labels are tensors before converting
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    try:
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    except Exception as e:
        print(f"Decoding error: {e}")
        return {"accuracy": 0.0}

    # Strip and normalize predictions and labels
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # Ensure consistent lengths
    min_len = min(len(decoded_preds), len(decoded_labels))
    decoded_preds = decoded_preds[:min_len]
    decoded_labels = decoded_labels[:min_len]

    # Calculate accuracy
    correct = sum(p == l for p, l in zip(decoded_preds, decoded_labels))
    accuracy = correct / len(decoded_labels) if len(decoded_labels) > 0 else 0.0

    return {"accuracy": accuracy}

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./",
    evaluation_strategy="epoch",
    save_strategy="no",                # <-- No intermediate checkpoints
    learning_rate=2e-4,
    per_device_train_batch_size=8,  # T5 requires smaller batches
    per_device_eval_batch_size=8,
    weight_decay=0.05,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=False,
    generation_max_length=2,    # Ensure short answer generation
    fp16=False,
    logging_dir="./second_baseline/logs",
    logging_steps=10,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    lr_scheduler_type="cosine",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_inputs,
    eval_dataset=valid_inputs,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_t5
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-124-b06efd624a98>:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.827500,0.805677,0.220126
2,0.791100,0.808466,0.236635
3,0.736400,0.859006,0.238994


TrainOutput(global_step=3819, training_loss=0.7859429999713843, metrics={'train_runtime': 4212.199, 'train_samples_per_second': 7.249, 'train_steps_per_second': 0.907, 'total_flos': 2.090838099964723e+16, 'train_loss': 0.7859429999713843, 'epoch': 3.0})

Save model (optional)

In [ ]:
trainer.save_model("./second_baseline/models/")
tokenizer.save_pretrained("./second_baseline/models/")
print("Model saved")

Model saved


Test on test data

In [ ]:
hf_test = Dataset.from_pandas(test_df)
test_inputs = Dataset.from_dict(preprocess_medqa_t5(hf_test))

results = trainer.evaluate(test_inputs)
with open("./results.csv", "w") as f:
    for key, value in results.items():
        f.write(f"{key}: {value}\n")
print(results)

100%|██████████| 1273/1273 [00:04<00:00, 273.78it/s]


{'eval_loss': 1.524336814880371, 'eval_accuracy': 0.32992930086410055, 'eval_runtime': 23.5308, 'eval_samples_per_second': 54.099, 'eval_steps_per_second': 3.4, 'epoch': 5.0}
